# 05 — Adaptif Saldirgana Karsi Savunma Testi (Adaptive Attacker)

**Cold-Start Sosyal-Grafik Sizintisi — Tehdit Modelini Guclendirme**

Makalenin savunma degerlendirmesi (04) savunmalari **sabit** bir saldirgana karsi olcer. Gercek saldirgan savunmayi bilir ve **en-iyi-yanitini** verir. Bu defter su tezi olcerek test eder:

> Cikti gurultusu, havuz genisletme ve cesitlendirme gibi savunmalar **sabit** saldirgana karsi guclu gorunur; ama **uyarlanan** saldirgan (gurultu/havuz icin sorgu-agregasyonu, cesitlendirme icin butce-genisletme) sizintiyi neredeyse savunmasiz seviyeye geri kazanir. **Rate-limiting**, geri-kazanimin sorgu butcesiyle buyumesini dogrudan sinirladigi icin ayakta kalan tek tamamlayicidir.

**Tasarim sadakati:** uretici + LightGCN + BPR + normalize gomme 02/02b ile **birebir**. Saldirgan-degeri yine cold-start baglam fallback'i (h=0): skor = sim(e_hedef, e_c). Savunmalar bu skor vektoru uzerinde tek bir yeniden-siralama/karistirma katmani olarak uygulanir; sabit ve adaptif saldirgan ayni katmana karsi olculur.

## 0. Kurulum

In [ ]:
!pip install -q networkx numpy pandas scipy torch matplotlib

In [ ]:
import os, json, time, math
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
def _in_colab():
    try:
        import google.colab; return True
    except Exception: return False
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT='/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT=os.path.abspath('./cold_start_leakage')
OUT_DIR=os.path.join(ROOT,'adaptive'); os.makedirs(OUT_DIR,exist_ok=True)
print("Cikti:",OUT_DIR,"| torch",torch.__version__)

## 1. Ayarlar

In [ ]:
MODE="quick"            # "quick" | "full"
BASELINE=dict(p_in=0.06,p_out=0.006,recip=0.49,homo=0.70,priv=0.30)
K=20
SIGMAS=(0.1,0.2,0.4)                    # gurultu savunmasi
Qs=(1,2,5,10,20,40)                     # saldirgan sorgu butcesi / rate-limit Q_max
LAMS=(0.1,0.5,0.9)                      # cesitlendirme (dusuk lam = agresif)
DIVBUDG=(1,2,3,5)                       # adaptif cesitlendirme: K'=b*K
POOLS=(0.3,0.5,0.7)                     # havuz seyreltme orani p
POOL_M=120                              # MMR aday havuzu boyutu
# raporlanan nominal calisma noktalari:
NOM=dict(sig=0.2,p=0.5,lam=0.1)
if MODE=="quick":
    SEEDS=[0,1,2]; GRAPHS=["sbm"]; EPOCHS=30
else:
    SEEDS=list(range(10)); GRAPHS=["sbm","lfr"]; EPOCHS=45
N_NODES=700
print(f"MODE={MODE} | {len(SEEDS)} seed x {len(GRAPHS)} grafik")

## 2. Uretici + model (02b ile birebir)

In [ ]:
class DS:
    def __init__(s,G,priv,comm,nl,X): s.G,s.private,s.community,s.node_list,s.features=G,priv,comm,nl,X
    def neigh(s,t): return set(s.G.predecessors(t))|set(s.G.successors(t))
def _orient(Gu,r,rng):
    D=nx.DiGraph(); D.add_nodes_from(Gu.nodes())
    for u,v in Gu.edges():
        if u==v: continue
        if rng.random()<r: D.add_edge(u,v); D.add_edge(v,u)
        else: (D.add_edge(u,v) if rng.random()<0.5 else D.add_edge(v,u))
    return D
def _feat(n,nc,comm,homo,fd,rng):
    mu=rng.normal(size=(nc,fd)); X=np.zeros((n,fd))
    for i in range(n): X[i]=homo*mu[comm[i]]+(1-homo)*rng.normal(size=fd)
    return ((X-X.mean(0))/(X.std(0)+1e-9)).astype(np.float32)
def _lab(n,priv,rng):
    pr={i:False for i in range(n)}
    for i in rng.choice(n,int(round(priv*n)),replace=False): pr[i]=True
    return pr
def make_sbm(n=700,nc=7,p_in=0.06,p_out=0.006,recip=0.49,homo=0.70,priv=0.30,fd=16,seed=0):
    rng=np.random.default_rng(seed); sizes=[n//nc]*nc
    for i in range(n-sum(sizes)): sizes[i]+=1
    P=np.full((nc,nc),p_out); np.fill_diagonal(P,p_in)
    Gu=nx.stochastic_block_model(sizes,P,seed=int(seed))
    comm={i:Gu.nodes[i]['block'] for i in Gu.nodes()}
    return DS(_orient(Gu,recip,rng),_lab(n,priv,rng),comm,list(range(n)),_feat(n,nc,comm,homo,fd,rng))
def make_lfr(n=700,recip=0.49,homo=0.70,priv=0.30,fd=16,seed=0,mu=0.1,**_):
    rng=np.random.default_rng(seed); Gu=None
    for off in range(8):
        try:
            Gu=nx.LFR_benchmark_graph(n=n,tau1=3,tau2=1.5,mu=mu,average_degree=10,min_community=30,seed=int(seed)+off,max_iters=500); break
        except Exception: continue
    if Gu is None: return None
    Gu=nx.Graph(Gu); Gu.remove_edges_from(nx.selfloop_edges(Gu)); Gu=nx.convert_node_labels_to_integers(Gu)
    raw={i:frozenset(Gu.nodes[i]["community"]) for i in Gu.nodes()}; uq={c:j for j,c in enumerate(set(raw.values()))}
    comm={i:uq[raw[i]] for i in Gu.nodes()}
    return DS(_orient(Gu,recip,rng),_lab(n,priv,rng),comm,list(range(n)),_feat(n,len(uq),comm,homo,fd,rng))
def norm_adj(G,n):
    und=G.to_undirected(); A=torch.zeros((n,n))
    for u,v in und.edges(): A[u,v]=1.0; A[v,u]=1.0
    A+=torch.eye(n); deg=A.sum(1); di=torch.pow(deg,-0.5); di[torch.isinf(di)]=0
    return torch.diag(di)@A@torch.diag(di)
class LightGCN(nn.Module):
    def __init__(s,n,dim=64,layers=3):
        super().__init__(); s.emb=nn.Embedding(n,dim); nn.init.normal_(s.emb.weight,std=0.1); s.layers=layers
    def forward(s,Ah):
        e=s.emb.weight; outs=[e]
        for _ in range(s.layers): e=Ah@e; outs.append(e)
        return torch.stack(outs,0).mean(0)
def train(model,Ah,pos,n,epochs=30,bs=1024,lr=0.01):
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5); pos=torch.tensor(pos,dtype=torch.long)
    for ep in range(epochs):
        perm=torch.randperm(pos.size(0))
        for i in range(0,pos.size(0),bs):
            idx=perm[i:i+bs]; u,p=pos[idx,0],pos[idx,1]; neg=torch.randint(0,n,(u.size(0),))
            E=model(Ah); su=(E[u]*E[p]).sum(1); sn=(E[u]*E[neg]).sum(1)
            loss=-torch.log(torch.sigmoid(su-sn)+1e-9).mean(); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return model(Ah).detach().numpy()

## 3. Savunmalar + sabit/adaptif saldirgan

- **Gurultu(sig):** skora N(0,sig) eklenir. Sabit = tek cekim (Q=1). Adaptif = Q cekimin ortalamasi (sifir-ortalamali gurultunun optimal tahmincisi -> Q buyudukce savunmasiza yaklasir).
- **Havuz(p):** her gosterilen slot p olasilikla rastgele dugumle degistirilir. Sabit = tek liste. Adaptif = Q liste boyunca frekans sayimi (gercek komsular tutarli gorunur).
- **Cesitlendirme(lam):** MMR yeniden-siralama (dusuk lam = agresif). Sabit = cesitlendirilmis ilk K. Adaptif = uzunluk K'=b*K listeyi ham alakaya gore yeniden siralayip ilk K.
- **Rate-limit:** Q_max ekseni; adaptif sizinti Q=Q_max'te okunur.

In [ ]:
def _prec_idx(idx,Nt,k): return len(set(list(idx)[:k])&Nt)/k
def _prec_top(scores,Nt,k):
    idx=np.argpartition(-scores,k)[:k]; return len(set(idx.tolist())&Nt)/k
def _mmr(s,sims,pool_idx,k,lam):
    rem=list(range(len(pool_idx))); sel=[]; mx=np.full(len(pool_idx),-1e9)
    while len(sel)<k and rem:
        if not sel: bi=max(rem,key=lambda i:s[pool_idx[i]])
        else: bi=max(rem,key=lambda i:lam*s[pool_idx[i]]-(1-lam)*mx[i])
        sel.append(bi); rem.remove(bi); mx=np.maximum(mx,sims[:,bi])
    return [pool_idx[i] for i in sel]

def evaluate(E,ds,seed=0,k=K):
    En=E/(np.linalg.norm(E,axis=1,keepdims=True)+1e-9); rng=np.random.default_rng(seed+777)
    n=En.shape[0]; targets=[t for t in ds.node_list if ds.private[t] and ds.G.degree(t)>=4]
    maxQ=max(Qs); acc={}
    def push(kk,v): acc.setdefault(kk,[]).append(v)
    for t in targets:
        Nt=ds.neigh(t)
        if not Nt: continue
        s=En@En[t]; s[t]=-1e9
        push("undef",_prec_top(s,Nt,k))
        # noise
        for sig in SIGMAS:
            running=np.zeros(n)
            for q in range(1,maxQ+1):
                running+=s+rng.normal(0,sig,n)
                if q==1: push(f"noise_static|sig={sig}",_prec_top(running.copy(),Nt,k))
                if q in Qs: push(f"noise_adapt|sig={sig}|Q={q}",_prec_top(running/q,Nt,k))
        # pool dilution
        base=np.argsort(-s)[:maxQ*k]
        for p in POOLS:
            tally=np.zeros(n)
            for q in range(1,maxQ+1):
                disp=[ (int(rng.integers(n)) if rng.random()<p else int(base[r])) for r in range(k) ]
                if q==1: push(f"pool_static|p={p}",_prec_idx(disp,Nt,k))
                for d in disp: tally[d]+=1
                if q in Qs: push(f"pool_adapt|p={p}|Q={q}",len(set(np.argsort(-tally)[:k].tolist())&Nt)/k)
        # diversification
        pool_idx=np.argsort(-s)[:POOL_M]; sims=(En[pool_idx]@En[pool_idx].T)
        for lam in LAMS:
            ml=_mmr(s,sims,pool_idx,max(DIVBUDG)*k,lam)
            push(f"div_static|lam={lam}",_prec_idx(ml,Nt,k))
            for b in DIVBUDG:
                sub=sorted(ml[:b*k],key=lambda c:-s[c])[:k]
                push(f"div_adapt|lam={lam}|b={b}",len(set(sub)&Nt)/k)
    return {kk:float(np.mean(v)) for kk,v in acc.items()}, len(acc["undef"])

## 4. ETA + cok-seed kosu

In [ ]:
def run_seed(gt,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    ds=(make_sbm(n=N_NODES,seed=seed,**BASELINE) if gt=="sbm" else make_lfr(n=N_NODES,seed=seed,homo=BASELINE["homo"],priv=BASELINE["priv"],recip=BASELINE["recip"]))
    if ds is None: return None,0
    n=len(ds.node_list); Ah=norm_adj(ds.G,n); pos=[(u,v) for u,v in ds.G.to_undirected().edges()]
    E=train(LightGCN(n),Ah,pos,n,epochs=EPOCHS)
    return evaluate(E,ds,seed=seed)

_t=time.time(); _r,_=run_seed(GRAPHS[0],SEEDS[0]); per=time.time()-_t
total=len(SEEDS)*len(GRAPHS)
print(f"~{per:.1f}s/seed | {total} seed-kosu | tahmini ~{per*total/60:.1f} dk")

In [ ]:
rows=[]; t0=time.time(); done=0
sig=NOM["sig"]
for gt in GRAPHS:
    for seed in SEEDS:
        res,nt=run_seed(gt,seed); done+=1
        if res is None: print(f"[atlandi] {gt} seed={seed}"); continue
        for kk,v in res.items(): rows.append(dict(graph=gt,seed=seed,condition=kk,value=v,n_targets=nt))
        pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"adaptive_long.csv"),index=False)
        st=res[f"noise_static|sig={sig}"]; ad=res[f"noise_adapt|sig={sig}|Q=40"]
        print(f"[{done}/{total}] {gt} seed={seed} undef={res['undef']:.3f} | noise sig={sig}: static={st:.3f} -> adaptiveQ40={ad:.3f}")
df=pd.DataFrame(rows); print(f"\nBitti: {len(df)} satir, {time.time()-t0:.1f}s")

## 5. Toplulastir + Claude'a yapistirilacak ozet

In [ ]:
def ci95(x):
    x=np.asarray(x,float); n=len(x)
    return 1.96*x.std(ddof=1)/math.sqrt(n) if n>1 else 0.0
df=pd.read_csv(os.path.join(OUT_DIR,"adaptive_long.csv"))
def agg(cond,graph):
    v=df[(df.condition==cond)&(df.graph==graph)]["value"].values
    return (float(np.mean(v)),ci95(v)) if len(v) else (float('nan'),float('nan'))
graphs=sorted(df.graph.unique()); sig=NOM["sig"]; p=NOM["p"]; lam=NOM["lam"]; Qm=max(Qs); bm=max(DIVBUDG)
L=["### 1) Adaptivite acigi — savunma nominal noktada (cold prec@20, mu+-CI; seed'ler arasi)",
   "(static = sabit saldirgan; adaptive = en-iyi-yanit tam butce; residual = undef - adaptive)","",
   "| grafik | savunma | knob | undef | static | adaptive | acik (ad-st) | kalan koruma (undef-ad) |",
   "|---|---|---|---|---|---|---|---|"]
for g in graphs:
    u=agg("undef",g)[0]
    for name,stat,adp,knob in [
        ("noise",f"noise_static|sig={sig}",f"noise_adapt|sig={sig}|Q={Qm}",f"sig={sig},Q={Qm}"),
        ("pool", f"pool_static|p={p}",     f"pool_adapt|p={p}|Q={Qm}",     f"p={p},Q={Qm}"),
        ("diversify",f"div_static|lam={lam}",f"div_adapt|lam={lam}|b={bm}",f"lam={lam},x{bm}")]:
        st=agg(stat,g); ad=agg(adp,g)
        L.append(f"| {g} | {name} | {knob} | {u:.3f} | {st[0]:.3f}+-{st[1]:.3f} | {ad[0]:.3f}+-{ad[1]:.3f} | "
                 f"{ad[0]-st[0]:+.3f} | {u-ad[0]:.3f} |")
L+=["","### 2) Gurultu x rate-limit (cold prec@20 mu) — adaptif sizinti Q_max'te kesilir","",
    "| grafik | sigma | "+" | ".join(f"Q_max={q}" for q in Qs)+" |",
    "|---|---|"+"|".join(["---"]*len(Qs))+"|"]
for g in graphs:
    for s_ in SIGMAS:
        L.append(f"| {g} | {s_} | "+" | ".join(f"{agg(f'noise_adapt|sig={s_}|Q={q}',g)[0]:.3f}" for q in Qs)+" |")
L+=["","### 3) Cesitlendirme: adaptif butce ile geri kazanim (cold prec@20 mu)","",
    "| grafik | lam | static |"+"".join(f" x{b} |" for b in DIVBUDG),
    "|---|---|---|"+"|".join(["---"]*len(DIVBUDG))+"|"]
for g in graphs:
    for l_ in LAMS:
        L.append(f"| {g} | {l_} | {agg(f'div_static|lam={l_}',g)[0]:.3f} |"+
                 "".join(f" {agg(f'div_adapt|lam={l_}|b={b}',g)[0]:.3f} |" for b in DIVBUDG))
meta=[f"# Cold-Start — Adaptive Attacker ozeti","",
      f"- MODE={MODE} | K={K} | seeds={len(SEEDS)} | graphs={graphs} | epochs={EPOCHS}",
      f"- undef cold prec@20: "+", ".join(f"{g}={agg('undef',g)[0]:.3f}" for g in graphs),""]
share="\n".join(meta+L)+"\n"
open(os.path.join(OUT_DIR,"SHARE_WITH_CLAUDE.md"),"w").write(share)
print("="*70); print(share); print("="*70)
print("Iki ===== arasini kopyalayip Claude'a gonder.")

## 6. (Opsiyonel) Figur — 3 panel

In [ ]:
import matplotlib.pyplot as plt
df=pd.read_csv(os.path.join(OUT_DIR,"adaptive_long.csv")); g=GRAPHS[0]
def mean_of(cond):
    v=df[(df.condition==cond)&(df.graph==g)]["value"].values; return float(np.mean(v)) if len(v) else np.nan
undef=mean_of("undef")
fig,ax=plt.subplots(1,3,figsize=(15,4))
for s_ in SIGMAS:
    ax[0].plot(Qs,[mean_of(f"noise_adapt|sig={s_}|Q={q}") for q in Qs],marker="o",label=f"sig={s_}")
ax[0].axhline(undef,ls="--",c="k",label="undefended"); ax[0].set_xscale("log")
ax[0].set_xlabel("saldirgan sorgu butcesi Q (=rate-limit Q_max)"); ax[0].set_ylabel("cold prec@20")
ax[0].set_title(f"Gurultu: adaptif geri-kazanim ({g})"); ax[0].legend()
for l_ in LAMS:
    ax[1].plot([0]+list(DIVBUDG),[mean_of(f"div_static|lam={l_}")]+[mean_of(f"div_adapt|lam={l_}|b={b}") for b in DIVBUDG],marker="s",label=f"lam={l_}")
ax[1].axhline(undef,ls="--",c="k"); ax[1].set_xlabel("adaptif butce carpani (0=static)"); ax[1].set_ylabel("cold prec@20")
ax[1].set_title("Cesitlendirme: butce ile geri kazanim"); ax[1].legend()
names=["noise","pool","diversify"]; sig=NOM["sig"]; p=NOM["p"]; lam=NOM["lam"]; Qm=max(Qs); bm=max(DIVBUDG)
stat=[mean_of(f"noise_static|sig={sig}"),mean_of(f"pool_static|p={p}"),mean_of(f"div_static|lam={lam}")]
adp=[mean_of(f"noise_adapt|sig={sig}|Q={Qm}"),mean_of(f"pool_adapt|p={p}|Q={Qm}"),mean_of(f"div_adapt|lam={lam}|b={bm}")]
x=np.arange(3); w=0.35
ax[2].bar(x-w/2,stat,w,label="static"); ax[2].bar(x+w/2,adp,w,label="adaptive")
ax[2].axhline(undef,ls="--",c="k",label="undefended"); ax[2].set_xticks(x); ax[2].set_xticklabels(names)
ax[2].set_ylabel("cold prec@20"); ax[2].set_title("Adaptivite acigi"); ax[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"adaptive_figs.png"),dpi=130); plt.show()
print("[kaydedildi]",os.path.join(OUT_DIR,"adaptive_figs.png"))